<a href="https://colab.research.google.com/github/aryalankit121/Data-Visualization-App/blob/main/Notebook_1_Prepare_2025_311_Dashboard_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 1: Prepare the 2025 311 Dashboard Dataset

## Purpose
Your original 311 file contains more rows and columns than a Streamlit dashboard needs. In this notebook, you will create a **smaller 2025 deployment dataset**.

You will:
1. Read the full 311 dataset.
2. Convert the request date to a real datetime value.
3. **Keep only requests from 2025.**
4. Create a few useful dashboard variables.
5. Select only the columns your dashboard needs.
6. Save the result as `311_2025_dashboard.csv`.
7. Check the new file size before using GitHub.

> **Important:** You are expected to edit the list of columns. Do not keep a column just because it appears in the starter code.


## 1. Import pandas and locate your file
Change `FULL_DATA_FILE` so that it matches the name/path of the full 311 file you downloaded.


In [31]:
import pandas as pd
from pathlib import Path

# EDIT THIS filename/path if needed
FULL_DATA_FILE = "311_data_notebook_1.csv"

# We will save the smaller file with this name
OUTPUT_FILE = "311_2025_dashboard.csv"


## 2. Read the full dataset
If your source file is a CSV, use the code below. If your instructor supplied a different file type, adjust the read command as directed.


In [32]:
df = pd.read_csv(FULL_DATA_FILE, low_memory=False)

print(f"Rows in full dataset: {len(df):,}")
print(f"Columns in full dataset: {len(df.columns):,}")
df.head()


Rows in full dataset: 518,841
Columns in full dataset: 20


,objectid,service_request_id,status,status_notes,service_name,service_code,agency_responsible,service_notice,requested_datetime,updated_datetime,expected_datetime,address,zipcode,media_url,lat,lon,subject,closed_datetime,gdb_geomattr_data,service_type
0,5056952,17346520,Closed,Issue Resolved,Graffiti Removal,SR-CL01,Community Life Improvement Program,7 Business Days,2025-01-01 00:00:34+00,2025-01-07 12:53:27+00,2025-01-10 00:00:00+00,1701 SPRING GARDEN ST,19130.0,NaN,39.963169,-75.166457,NaN,2025-01-07 12:53:24+00,NaN,Graffiti Removal
1,5056953,17346521,Closed,Issue Resolved,Graffiti Removal,SR-CL01,Community Life Improvement Program,7 Business Days,2025-01-01 00:03:51+00,2025-01-07 12:52:31+00,2025-01-10 00:00:00+00,1901 SPRING GARDEN ST,19130.0,NaN,39.963625,-75.169500,NaN,2025-01-07 12:52:28+00,NaN,Graffiti Removal
2,5056954,17346523,Closed,NaN,Rubbish/Recyclable Material Collection,SR-ST03,Streets Department,2 Business Days,2025-01-01 00:06:29+00,2025-03-03 14:28:52+00,2025-01-03 00:00:00+00,5902 JEFFERSON ST,19151.0,https://d17aqltn7cihbm.cloudfront.net/uploads/...,39.978809,-75.239265,NaN,2025-03-03 14:25:15+00,NaN,Recyclables Collection
3,5056955,17346524,Closed,Issue Resolved,Graffiti Removal,SR-CL01,Community Life Improvement Program,7 Business Days,2025-01-01 00:06:47+00,2025-01-07 12:48:56+00,2025-01-10 00:00:00+00,1921 SPRING GARDEN ST,19130.0,NaN,39.963703,-75.170116,NaN,2025-01-07 12:48:46+00,NaN,Graffiti Removal
4,5056956,17346525,Closed,NaN,Rubbish/Recyclable Material Collection,SR-ST03,Streets Department,2 Business Days,2025-01-01 00:07:43+00,2025-01-03 11:30:26+00,2025-01-03 00:00:00+00,6739 RUTLAND ST,19149.0,NaN,40.043401,-75.072138,NaN,2025-01-03 11:25:48+00,NaN,Recyclables Collection


## 3. Look at the available columns
Before deleting anything, see what is available. Your dashboard should keep only fields that support the questions and charts you want to show.


In [33]:
print(df.columns.tolist())


['objectid', 'service_request_id', 'status', 'status_notes', 'service_name', 'service_code', 'agency_responsible', 'service_notice', 'requested_datetime', 'updated_datetime', 'expected_datetime', 'address', 'zipcode', 'media_url', 'lat', 'lon', 'subject', 'closed_datetime', 'gdb_geomattr_data', 'service_type']


## 4. Convert the request timestamp and keep ONLY 2025
The dashboard for this project must use **2025 requests only**.

`errors="coerce"` changes invalid date values to missing values instead of crashing the notebook.


In [34]:
df["requested_datetime"] = pd.to_datetime(
    df["requested_datetime"],
    errors="coerce",
    utc=True
)

# Required: keep only 2025 requests
df_2025 = df[df["requested_datetime"].dt.year == 2025].copy()

print(f"2025 rows: {len(df_2025):,}")
print("First request:", df_2025["requested_datetime"].min())
print("Last request: ", df_2025["requested_datetime"].max())


2025 rows: 518,841
First request: 2025-01-01 00:00:34+00:00
Last request:  2025-12-31 23:51:10+00:00


## 5. Create variables that will be useful in the dashboard
The most important new field is `request_date`. Streamlit will later use it to let the user choose a **start date** and **end date**.

We will also create `month` and `day_of_week`. You may use these in your dashboard, or you may remove them later if you do not need them.


In [35]:
# A date-only field is convenient for dashboard filtering
df_2025["request_date"] = df_2025["requested_datetime"].dt.date

# Optional but useful dashboard variables
df_2025["month"] = df_2025["requested_datetime"].dt.month_name()
df_2025["day_of_week"] = df_2025["requested_datetime"].dt.day_name()

print("Dashboard date range:")
print(df_2025["request_date"].min(), "through", df_2025["request_date"].max())


Dashboard date range:
2025-01-01 through 2025-12-31


## 6. Optional: Calculate resolution time
If you want to analyze how long requests take to close, create `resolution_days`.

If you do **not** plan to use resolution time in your dashboard, you can skip this section.


In [36]:
if "closed_datetime" in df_2025.columns:
    df_2025["closed_datetime"] = pd.to_datetime(
        df_2025["closed_datetime"],
        errors="coerce",
        utc=True
    )

    df_2025["resolution_days"] = (
        df_2025["closed_datetime"] - df_2025["requested_datetime"]
    ).dt.total_seconds() / 86400

    print(df_2025["resolution_days"].describe())


count    485832.000000
mean         20.043554
std          59.484267
min           0.000000
25%           0.000000
50%           0.000000
75%           4.721973
max         628.013044
Name: resolution_days, dtype: float64


## 7. Choose the columns for YOUR dashboard
The list below is only a starting point. **Edit it.**

Keep fields needed for your charts, filters, metrics, or map. Remove fields you do not use. Large text/media fields such as `media_url`, `status_notes`, `subject`, or `gdb_geomattr_data` usually do not belong in this dashboard dataset unless you have a specific reason to use them.

If you created `resolution_days` and want to use it, add it to the list.


In [37]:
# EDIT THIS LIST based on the dashboard you plan to build
columns_to_keep = [
    # Identifiers & Categorization
    "service_request_id",
    "service_type",
    "service_name",
    "agency_responsible",
    "status",

    # Dates & Time Tracking
    "request_date",
    "requested_datetime",
    "closed_datetime",
    "expected_datetime",
    "month",
    "day_of_week",
    "resolution_days",  # Include if you ran Step 6

    # Location Data
    "zipcode",
    "lat",
    "lon",
]

# Keep only columns that actually exist in the source file.
# This makes the starter notebook a little more forgiving if column names vary.
columns_to_keep = [c for c in columns_to_keep if c in df_2025.columns]

dashboard_df = df_2025[columns_to_keep].copy()

print(f"Dashboard rows: {len(dashboard_df):,}")
print(f"Dashboard columns: {len(dashboard_df.columns)}")
dashboard_df.head()


Dashboard rows: 518,841
Dashboard columns: 15


,service_request_id,service_type,service_name,agency_responsible,status,request_date,requested_datetime,closed_datetime,expected_datetime,month,day_of_week,resolution_days,zipcode,lat,lon
0,17346520,Graffiti Removal,Graffiti Removal,Community Life Improvement Program,Closed,2025-01-01,2025-01-01 00:00:34+00:00,2025-01-07 12:53:24+00:00,2025-01-10 00:00:00+00,January,Wednesday,6.536690,19130.0,39.963169,-75.166457
1,17346521,Graffiti Removal,Graffiti Removal,Community Life Improvement Program,Closed,2025-01-01,2025-01-01 00:03:51+00:00,2025-01-07 12:52:28+00:00,2025-01-10 00:00:00+00,January,Wednesday,6.533762,19130.0,39.963625,-75.169500
2,17346523,Recyclables Collection,Rubbish/Recyclable Material Collection,Streets Department,Closed,2025-01-01,2025-01-01 00:06:29+00:00,2025-03-03 14:25:15+00:00,2025-01-03 00:00:00+00,January,Wednesday,61.596366,19151.0,39.978809,-75.239265
3,17346524,Graffiti Removal,Graffiti Removal,Community Life Improvement Program,Closed,2025-01-01,2025-01-01 00:06:47+00:00,2025-01-07 12:48:46+00:00,2025-01-10 00:00:00+00,January,Wednesday,6.529155,19130.0,39.963703,-75.170116
4,17346525,Recyclables Collection,Rubbish/Recyclable Material Collection,Streets Department,Closed,2025-01-01,2025-01-01 00:07:43+00:00,2025-01-03 11:25:48+00:00,2025-01-03 00:00:00+00,January,Wednesday,2.470891,19149.0,40.043401,-75.072138


## 8. Check missing values and data types
Do a quick check before saving. This is not another full EDA; you are simply making sure the deployment data looks reasonable.


In [38]:
print(dashboard_df.dtypes)
print("Missing values:")
print(dashboard_df.isna().sum().sort_values(ascending=False))


service_request_id                  int64
service_type                       object
service_name                       object
agency_responsible                 object
status                             object
request_date                       object
requested_datetime    datetime64[ns, UTC]
closed_datetime       datetime64[ns, UTC]
expected_datetime                  object
month                              object
day_of_week                        object
resolution_days                   float64
zipcode                           float64
lat                               float64
lon                               float64
dtype: object
Missing values:
expected_datetime     294352
zipcode               277255
lat                   273033
lon                   273033
closed_datetime        33009
resolution_days        33009
service_type              21
requested_datetime         0
request_date               0
status                     0
agency_responsible         0
service_name         

## 9. Save the reduced dataset
This is the file that your second notebook and your Streamlit application will use.


In [39]:
dashboard_df.to_csv(OUTPUT_FILE, index=False)
print(f"Saved: {OUTPUT_FILE}")


Saved: 311_2025_dashboard.csv


## 10. Check the saved file size
For this assignment, keep the dashboard data file **under 25 MB** so it is easy to work with in GitHub.

If the file is still too large, go back to your `columns_to_keep` list and remove fields your dashboard does not need. You may also discuss other reasonable reductions with your instructor.


In [40]:
file_size_mb = Path(OUTPUT_FILE).stat().st_size / (1024 * 1024)
print(f"File size: {file_size_mb:.2f} MB")

if file_size_mb < 25:
    print("Good: the file is under 25 MB.")
else:
    print("The file is still too large for this assignment. Reduce it further.")


File size: 95.95 MB
The file is still too large for this assignment. Reduce it further.


## 11. Final verification
Your reduced dataset should:
- contain **2025 only**;
- include `request_date`;
- include only fields that have a purpose in your dashboard;
- be under **25 MB**;
- be saved as `311_2025_dashboard.csv`.

You are now ready for **Notebook 2: Prototype Your Dashboard Visualizations**.
